# Lateral Double Quantum Dot Results and Plots

This notebook summarizes the 3D lateral double quantum dot run and collects presentation-ready figures for the thesis document and final presentation.

Run folder:
`runs/Double_Quantum_Dot_3D__20260429_004136`

In [44]:
RUN_ROOT = PROJECT_ROOT / "runs" / "Double_Quantum_Dot_3D_from_PHIDL__20260606_043720__geometry_check_no_padding"

In [46]:
from pathlib import Path
import importlib
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "qpu-design-automation-toolkit" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.nextnanopp_tools as nnt

nnt = importlib.reload(nnt)
plot_gate_bandedge_density_3d = nnt.plot_gate_bandedge_density_3d

RUN_ROOT = PROJECT_ROOT / "runs" / "Double_Quantum_Dot_3D__20260429_004136"
SIMULATION_LAYOUT_JSON = PROJECT_ROOT / "data" / "gds" / "double_dot_from_phidl_simulation_layout.json"
ARTIFACT_DIR = PROJECT_ROOT / "notebooks" / "analysis" / "artifacts" / "lateral_double_quantum_dot_results_and_plots"

TARGET_Z_NM = -4.0
BIAS = 0

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RUN_ROOT

PosixPath('/Users/robertjovanov/code/qpu-design-automation-toolkit/runs/Double_Quantum_Dot_3D__20260429_004136')

## Gates, HH Band Edge, and Hole Density

The target depth is `z = -4 nm`. The reusable loaders extract the nearest available mesh plane and report the actual plane coordinate in the figure metadata/title.

For this presentation-style view, the HH band edge and hole density are plotted as 3D profiles: the value on the `x-y` plane becomes the plotted height. The gate layer is drawn from the simulated `Structure/contacts.vtr` contact-index output as a flat top-view projection above the two profile surfaces.

In [45]:
fig_hh_density_gates = plot_gate_bandedge_density_3d(
    RUN_ROOT,
    SIMULATION_LAYOUT_JSON,
    z_nm=TARGET_Z_NM,
    bias=BIAS,
    bandedge_variable="HH",
    density_quantity="density_hole",
    density_variable=None,
    density_log10=False,
    density_threshold_percentile=None,
    gate_source="contacts_projection",
    bandedge_colorscale="Greens",
    title="Lateral double quantum dot: HH band edge and hole density under top gates",
)

fig_hh_density_gates

In [ ]:
html_path = ARTIFACT_DIR / "hh_bandedge_hole_density_gates_z_minus4nm.html"
fig_hh_density_gates.write_html(html_path, include_plotlyjs="cdn")
html_path

## HH0 Ground-State Peak Depths

Extract the two laterally separated maxima of the ground-state HH0 probability density and report the z-coordinate where each peak occurs.

In [47]:
HH0_PROBABILITY_VTR = (
    RUN_ROOT
    / "bias_00000"
    / "Quantum"
    / "c-Ge_QW"
    / "HH"
    / "probability_shift_k00000_0001.vtr"
)
HH0_BANDEDGES_VTR = RUN_ROOT / "bias_00000" / "bandedges.vtr"

hh0_probability_peaks = nnt.find_probability_peaks(
    HH0_PROBABILITY_VTR,
    variable="Psi^2_1",
    n_peaks=2,
    min_lateral_separation_nm=50.0,
)

display(hh0_probability_peaks[["peak", "x_nm", "y_nm", "z_nm", "probability"]])
print("HH0 peak z coordinates (nm):", hh0_probability_peaks["z_nm"].tolist())

,peak,x_nm,y_nm,z_nm,probability
0,1,1310.0,50.0,-4.0,0.001916
1,2,1170.0,50.0,-4.0,0.001916


HH0 peak z coordinates (nm): [-4.0, -4.0]


In [48]:
hh0_peak_z_values = sorted(hh0_probability_peaks["z_nm"].unique())
hh0_peak_z_nm = float(hh0_peak_z_values[0])

if len(hh0_peak_z_values) > 1:
    print(f"HH0 peaks occur at multiple z coordinates: {hh0_peak_z_values}. Plotting z={hh0_peak_z_nm:g} nm.")
else:
    print(f"Plotting HH0 probability density at peak z={hh0_peak_z_nm:g} nm.")

fig_hh0_probability_xy = nnt.plot_vtr_slice_interactive(
    HH0_PROBABILITY_VTR,
    variable="Psi^2_1",
    slice_axis="z",
    slice_value=hh0_peak_z_nm,
    title=f"HH0 probability density x-y slice at z={hh0_peak_z_nm:g} nm",
    colorscale="Reds",
)

Plotting HH0 probability density at peak z=-4 nm.


## HH Band Edge at the HH0 Peak Depth

Plot the HH band edge on the same `x-y` plane where the HH0 probability peaks were found.

In [49]:
fig_hh_bandedge_peak_depth = nnt.plot_vtr_slice_interactive(
    HH0_BANDEDGES_VTR,
    variable="HH",
    slice_axis="z",
    slice_value=hh0_peak_z_nm,
    title=f"HH band edge at z={hh0_peak_z_nm:g} nm",
    colorscale="Greens",
)

## HH Band Edge at the HH0 Peak Coordinates

Sample the nearest HH band-edge grid point at each HH0 probability peak coordinate.

In [50]:
hh_bandedge_at_hh0_peaks = nnt.sample_vtr_plane_at_points(
    HH0_BANDEDGES_VTR,
    variable="HH",
    slice_axis="z",
    slice_value=hh0_peak_z_nm,
    points=hh0_probability_peaks,
)

hh_bandedge_at_hh0_peaks[[
    "point",
    "requested_x_nm",
    "requested_y_nm",
    "nearest_x_nm",
    "nearest_y_nm",
    "z_nm",
    "HH[eV]",
]]

,point,requested_x_nm,requested_y_nm,nearest_x_nm,nearest_y_nm,z_nm,HH[eV]
0,1,1310.0,50.0,1310.0,50.0,-4.0,0.009262
1,2,1170.0,50.0,1170.0,50.0,-4.0,0.009262


## HH Band Edge Extrema at z = -4 nm

Find the two laterally separated maxima of the HH band edge on the `z = -4 nm` plane.

In [51]:
HH_BANDEDGE_EXTREMA_Z_NM = -4.0

hh_bandedge_extrema = nnt.find_vtr_plane_extrema(
    HH0_BANDEDGES_VTR,
    variable="HH",
    slice_axis="z",
    slice_value=HH_BANDEDGE_EXTREMA_Z_NM,
    kind="max",
    n_extrema=2,
    min_lateral_separation_nm=50.0,
)

hh_bandedge_extrema[["extremum", "kind", "x_nm", "y_nm", "z_nm", "HH[eV]"]]

,extremum,kind,x_nm,y_nm,z_nm,HH[eV]
0,1,max,1175.0,60.0,-4.0,0.00949
1,2,max,1305.0,60.0,-4.0,0.00949
